# Beans Good/Bad Classifier
Run cells top to bottom with **Shift+Enter**. Edit the paths in cell 1 first.

In [ ]:
# 1. Imports and config
import os, glob
from PIL import Image, ImageOps
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt

# EDIT THIS: folder where you extracted the Mendeley dataset
RAW_DIR = r"C:\Users\YourName\Desktop\beans_raw"

# Resized copies get saved here (created automatically)
DATA_DIR = "data"

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 15
SEED = 42
print("TensorFlow", tf.__version__)

In [ ]:
# 2. Look at what the dataset folder contains
for root, dirs, files in os.walk(RAW_DIR):
    print(root, "->", len(files), "files")

If the folders aren't named exactly `good` and `bad`, change `CLASS_FOLDERS` in the next cell to match what you saw above.

In [ ]:
# 3. Resize images into data/good and data/bad (one-time, shrinks 2.4 GB a lot)
CLASS_FOLDERS = {"good": "good", "bad": "bad"}   # {output name: folder name in RAW_DIR}
exts = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")

for out_name, raw_name in CLASS_FOLDERS.items():
    os.makedirs(os.path.join(DATA_DIR, out_name), exist_ok=True)
    files = []
    for e in exts:
        files += glob.glob(os.path.join(RAW_DIR, "**", raw_name, "**", e), recursive=True)
    files = sorted(set(files))
    for i, f in enumerate(files):
        img = ImageOps.exif_transpose(Image.open(f)).convert("RGB").resize((256, 256))
        img.save(os.path.join(DATA_DIR, out_name, f"{out_name}_{i:04d}.jpg"), quality=90)
    print(out_name, ":", len(files), "images resized")

In [ ]:
# 4. Load data (80% train / 20% validation)
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="training", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="binary")
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="validation", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="binary")

class_names = train_ds.class_names
print("Classes:", class_names, "(0 =", class_names[0], ", 1 =", class_names[1], ")")

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(200).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

In [ ]:
# 5. Peek at a few training images to sanity-check labels
plt.figure(figsize=(8, 8))
for images, labels in train_ds.take(1):
    for i in range(min(9, len(images))):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i][0])])
        plt.axis("off")
plt.show()

In [ ]:
# 6. Augmentation + model (MobileNetV2, frozen base)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.3),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2),
])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
# 7. Train
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

In [ ]:
# 8. Plot accuracy and loss
acc, val_acc = history.history["accuracy"], history.history["val_accuracy"]
loss, val_loss = history.history["loss"], history.history["val_loss"]

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(acc, label="train"); plt.plot(val_acc, label="validation")
plt.title("Accuracy"); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(loss, label="train"); plt.plot(val_loss, label="validation")
plt.title("Loss"); plt.legend()
plt.show()

In [ ]:
# 9. Confusion matrix on the validation set (where is it getting things wrong?)
import numpy as np
y_true, y_pred = [], []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true += labels.numpy().flatten().astype(int).tolist()
    y_pred += (preds.flatten() > 0.5).astype(int).tolist()

y_true, y_pred = np.array(y_true), np.array(y_pred)
cm = np.zeros((2, 2), dtype=int)
for t, p in zip(y_true, y_pred):
    cm[t, p] += 1
print("Rows = actual, columns = predicted  (order:", class_names, ")")
print(cm)
print("Validation accuracy:", (y_true == y_pred).mean())

In [ ]:
# 10. Save the model
model.save("bean_classifier.keras")
print("Saved bean_classifier.keras")

In [ ]:
# 11. Test on a single new image (e.g. one of YOUR own bean photos)
test_path = r"C:\Users\YourName\Desktop\my_bean_photo.jpg"   # EDIT THIS

img = tf.keras.utils.load_img(test_path, target_size=IMG_SIZE)
arr = tf.expand_dims(tf.keras.utils.img_to_array(img), 0)
p = float(model.predict(arr, verbose=0)[0][0])
label = class_names[int(p > 0.5)]
conf = p if p > 0.5 else 1 - p
plt.imshow(img); plt.axis("off"); plt.title(f"{label} ({conf:.0%})"); plt.show()